# Exploración de los datos desde cero (módulo 3)

Este notebook lee los **4 archivos CSV originales** de Kaggle, muestra cómo es cada tabla y aplica pruebas sencillas que indican que los datos son **sintéticos**, no un registro real de turistas.

Todo el código está aquí, sin funciones escondidas, para que se pueda leer y modificar paso a paso. Las cifras coinciden con las del informe (secciones 1 y 3).

**Antes de empezar:** los datos originales no se suben a git. Hay que descargar el zip de [Kaggle](https://www.kaggle.com/datasets/amanmehra23/travel-recommendation-dataset) y ponerlo (o descomprimirlo) en `modulo3_recomendacion/data/raw/`.

## 1. Leer los archivos

In [1]:
from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 200, "display.max_columns", 20)

# Buscar la carpeta data/raw del módulo 3, desde notebooks/ o desde la raíz del repo
for base in (Path.cwd(), *Path.cwd().parents):
    for cand in (base / "data" / "raw", base / "modulo3_recomendacion" / "data" / "raw"):
        if cand.exists():
            RAW = cand
            break
    else:
        continue
    break
else:
    raise FileNotFoundError("No encuentro modulo3_recomendacion/data/raw: descarga el zip de Kaggle y ponlo ahí.")

ARCHIVOS = {"destinos": "Expanded_Destinations.csv", "usuarios": "Final_Updated_Expanded_Users.csv",
            "resenas": "Final_Updated_Expanded_Reviews.csv", "historial": "Final_Updated_Expanded_UserHistory.csv"}

def leer(nombre):
    ruta = RAW / ARCHIVOS[nombre]
    if ruta.exists():
        return pd.read_csv(ruta)
    with zipfile.ZipFile(RAW / "archive.zip") as z:        # si solo está el zip, se lee directo del zip
        interno = next(n for n in z.namelist() if n.endswith(ARCHIVOS[nombre]))
        return pd.read_csv(z.open(interno))

destinos, usuarios, resenas, historial = (leer(n) for n in ARCHIVOS)
for nombre, df in zip(ARCHIVOS, (destinos, usuarios, resenas, historial)):
    print(f"{nombre:<10} {df.shape[0]:>5} filas × {df.shape[1]} columnas: {list(df.columns)}")

destinos    1000 filas × 6 columnas: ['DestinationID', 'Name', 'State', 'Type', 'Popularity', 'BestTimeToVisit']
usuarios     999 filas × 7 columnas: ['UserID', 'Name', 'Email', 'Preferences', 'Gender', 'NumberOfAdults', 'NumberOfChildren']
resenas      999 filas × 5 columnas: ['ReviewID', 'DestinationID', 'UserID', 'Rating', 'ReviewText']
historial    999 filas × 5 columnas: ['HistoryID', 'UserID', 'DestinationID', 'VisitDate', 'ExperienceRating']


## 2. Cómo es cada tabla

- **destinos:** el catálogo. Un ID, el nombre del lugar, el estado de India, el tipo, una puntuación de popularidad y la mejor época para ir.
- **usuarios:** un ID, nombre y correo, las preferencias que declararon, el género y cuántos adultos y niños viajan.
- **resenas:** qué usuario reseñó qué destino, con una nota de 1 a 5 y un texto.
- **historial:** qué usuario visitó qué destino, en qué fecha y con qué nota de la experiencia.

In [2]:
destinos.head(10)

,DestinationID,Name,State,Type,Popularity,BestTimeToVisit
0,1,Taj Mahal,Uttar Pradesh,Historical,8.691906,Nov-Feb
1,2,Goa Beaches,Goa,Beach,8.605032,Nov-Mar
2,3,Jaipur City,Rajasthan,City,9.225372,Oct-Mar
3,4,Kerala Backwaters,Kerala,Nature,7.977386,Sep-Mar
4,5,Leh Ladakh,Jammu and Kashmir,Adventure,8.399822,Apr-Jun
5,6,Taj Mahal,Uttar Pradesh,Historical,7.648950,Nov-Feb
6,7,Goa Beaches,Goa,Beach,9.145068,Nov-Mar
7,8,Jaipur City,Rajasthan,City,9.458705,Oct-Mar
8,9,Kerala Backwaters,Kerala,Nature,9.260450,Sep-Mar
9,10,Leh Ladakh,Jammu and Kashmir,Adventure,8.811952,Apr-Jun


In [3]:
usuarios.head(10)

,UserID,Name,Email,Preferences,Gender,NumberOfAdults,NumberOfChildren
0,1,Kavya,kavya@example.com,"Beaches, Historical",Female,1,0
1,2,Rohan,rohan@example.com,"Nature, Adventure",Male,2,2
2,3,Kavya,kavya@example.com,"City, Historical",Female,2,0
3,4,Anika,anika@example.com,"Beaches, Historical",Female,1,0
4,5,Tanvi,tanvi@example.com,"Nature, Adventure",Female,2,2
5,6,Karan,karan@example.com,"City, Historical",Male,2,0
6,7,Rohan,rohan@example.com,"Beaches, Historical",Male,2,1
7,8,Vivaan,vivaan@example.com,"Nature, Adventure",Male,2,2
8,9,Riya,riya@example.com,"City, Historical",Female,2,0
9,10,Rohan,rohan@example.com,"Beaches, Historical",Male,2,2


In [4]:
resenas.head(10)

,ReviewID,DestinationID,UserID,Rating,ReviewText
0,1,178,327,2,Incredible monument!
1,2,411,783,1,Loved the beaches!
2,3,927,12,2,A historical wonder
3,4,358,959,3,Incredible monument!
4,5,989,353,2,Loved the beaches!
5,6,473,408,4,A historical wonder
6,7,977,735,3,Incredible monument!
7,8,865,664,1,Loved the beaches!
8,9,698,330,2,A historical wonder
9,10,919,109,5,Incredible monument!


In [5]:
historial.head(10)

,HistoryID,UserID,DestinationID,VisitDate,ExperienceRating
0,1,525,760,2024-01-01,3
1,2,184,532,2024-02-15,5
2,3,897,786,2024-03-20,2
3,4,470,660,2024-01-01,1
4,5,989,389,2024-02-15,4
5,6,266,394,2024-03-20,3
6,7,399,953,2024-01-01,2
7,8,869,901,2024-02-15,4
8,9,580,890,2024-03-20,1
9,10,876,727,2024-01-01,4


In [6]:
# Calidad: datos faltantes, IDs repetidos e interacciones que apunten a algo que no existe
for nombre, df, id_col in (("destinos", destinos, "DestinationID"), ("usuarios", usuarios, "UserID"),
                           ("resenas", resenas, "ReviewID"), ("historial", historial, "HistoryID")):
    print(f"{nombre:<10} faltantes: {int(df.isna().sum().sum())}  IDs repetidos: {int(df[id_col].duplicated().sum())}")
for nombre, df in (("resenas", resenas), ("historial", historial)):
    print(f"{nombre:<10} usuarios inexistentes: {int((~df.UserID.isin(usuarios.UserID)).sum())}  "
          f"destinos inexistentes: {int((~df.DestinationID.isin(destinos.DestinationID)).sum())}")

destinos   faltantes: 0  IDs repetidos: 0
usuarios   faltantes: 0  IDs repetidos: 0
resenas    faltantes: 0  IDs repetidos: 0
historial  faltantes: 0  IDs repetidos: 0
resenas    usuarios inexistentes: 0  destinos inexistentes: 0
historial  usuarios inexistentes: 0  destinos inexistentes: 0


## 3. La tabla de interacciones (reseñas + historial)

Una **interacción** es un momento en que un usuario tuvo contacto con un destino. Unimos reseñas e historial en una sola tabla (decisión 2 del informe).

In [7]:
inter = pd.concat([
    resenas.rename(columns={"Rating": "calificacion"})[["UserID", "DestinationID", "calificacion"]].assign(fuente="reseña"),
    historial.rename(columns={"ExperienceRating": "calificacion"})[["UserID", "DestinationID", "calificacion"]].assign(fuente="historial"),
], ignore_index=True)
inter = inter.merge(destinos[["DestinationID", "Name", "Type"]], on="DestinationID")
print("Interacciones:", len(inter), "| usuarios con ≥ 1:", inter.UserID.nunique(), "| destinos con ≥ 1:", inter.DestinationID.nunique())
print(f"Densidad: {len(inter)} / (999 × 1000) = {100 * len(inter) / (999 * 1000):.2f} %")
inter.head(10)

Interacciones: 1998 | usuarios con ≥ 1: 858 | destinos con ≥ 1: 866
Densidad: 1998 / (999 × 1000) = 0.20 %


,UserID,DestinationID,calificacion,fuente,Name,Type
0,327,178,2,reseña,Jaipur City,City
1,783,411,1,reseña,Taj Mahal,Historical
2,12,927,2,reseña,Goa Beaches,Beach
3,959,358,3,reseña,Jaipur City,City
4,353,989,2,reseña,Kerala Backwaters,Nature
5,408,473,4,reseña,Jaipur City,City
6,735,977,3,reseña,Goa Beaches,Beach
7,664,865,1,reseña,Leh Ladakh,Adventure
8,330,698,2,reseña,Jaipur City,City
9,109,919,5,reseña,Kerala Backwaters,Nature


## 4. Pruebas de que los datos son sintéticos

Cada prueba tiene la misma estructura: **la idea** (qué esperaríamos en datos reales), **el código** y **cómo leer el resultado**. Las primeras son comprobaciones directas; las demás son pruebas de hipótesis con su H₀, su H₁ y su p-valor (si p-valor < 0,05, se rechaza H₀).

### 4.1 Los 1000 destinos son 5 lugares que se repiten

**Idea:** un catálogo real tendría 1000 destinos distintos. Si el nombre del lugar depende solo de la posición del ID en un ciclo de 5 (IDs 1, 6, 11, … siempre el mismo lugar), el catálogo fue fabricado copiando 5 lugares.

In [8]:
print("Nombres distintos:", destinos.Name.nunique())
pd.crosstab((destinos.DestinationID - 1) % 5, destinos.Name).rename_axis(index="(ID − 1) mod 5")

Nombres distintos: 5


Name,Goa Beaches,Jaipur City,Kerala Backwaters,Leh Ladakh,Taj Mahal
(ID − 1) mod 5,,,,,
0,0,0,0,0,200
1,200,0,0,0,0
2,0,200,0,0,0
3,0,0,200,0,0
4,0,0,0,200,0


In [9]:
# Dentro de cada lugar, ¿cambia algo además de Popularity?
destinos.groupby("Name").agg(ids=("DestinationID", "size"), tipos=("Type", "nunique"), estados=("State", "nunique"),
                             temporadas=("BestTimeToVisit", "nunique"),
                             popularity_min=("Popularity", "min"), popularity_max=("Popularity", "max")).round(2)

,ids,tipos,estados,temporadas,popularity_min,popularity_max
Name,,,,,,
Goa Beaches,200,1,1,1,7.50,9.50
Jaipur City,200,1,1,1,7.51,9.50
Kerala Backwaters,200,1,1,1,7.53,9.49
Leh Ladakh,200,1,1,1,7.53,9.48
Taj Mahal,200,1,1,1,7.51,9.48


**Cómo leerlo:** cada fila del cruce tiene un solo lugar con 200 IDs, así que el lugar queda **totalmente determinado** por (ID − 1) mod 5. Tipo, estado y temporada no cambian dentro de un lugar; solo `Popularity` varía, y lo hace en el mismo rango (≈ 7,5–9,5) en los cinco.

### 4.2 Ciclos exactos en usuarios, reseñas e historial

**Idea:** en datos reales, las preferencias de una persona, el texto de una reseña o la fecha de un viaje no dependen de su número de fila.

In [10]:
def ciclo(df, id_col, col, k):
    """¿El valor de 'col' queda determinado por (ID − 1) mod k?"""
    return bool((df.groupby((df[id_col] - 1) % k)[col].nunique() == 1).all())

pd.DataFrame([
    ("usuarios", "Preferences", usuarios.Preferences.nunique(), 3, ciclo(usuarios, "UserID", "Preferences", 3)),
    ("usuarios", "Name", usuarios.Name.nunique(), 20, ciclo(usuarios, "UserID", "Name", 20)),
    ("usuarios", "Email", usuarios.Email.nunique(), 20, ciclo(usuarios, "UserID", "Email", 20)),
    ("resenas", "ReviewText", resenas.ReviewText.nunique(), 3, ciclo(resenas, "ReviewID", "ReviewText", 3)),
    ("historial", "VisitDate", historial.VisitDate.nunique(), 3, ciclo(historial, "HistoryID", "VisitDate", 3)),
], columns=["tabla", "columna", "valores distintos", "ciclo k", "¿determinada por (ID − 1) mod k?"])

,tabla,columna,valores distintos,ciclo k,¿determinada por (ID − 1) mod k?
0,usuarios,Preferences,3,3,True
1,usuarios,Name,20,20,False
2,usuarios,Email,20,20,False
3,resenas,ReviewText,3,3,True
4,historial,VisitDate,3,3,True


In [11]:
print(historial.VisitDate.value_counts().sort_index(), "\n")
print(resenas.ReviewText.value_counts(), "\n")
print(usuarios.Preferences.value_counts())

VisitDate
2024-01-01    333
2024-02-15    333
2024-03-20    333
Name: count, dtype: int64 

ReviewText
Incredible monument!    333
Loved the beaches!      333
A historical wonder     333
Name: count, dtype: int64 

Preferences
Beaches, Historical    333
Nature, Adventure      333
City, Historical       333
Name: count, dtype: int64


**Cómo leerlo:** hay solo 3 fechas de visita (333 visitas en cada una), 3 textos de reseña, 3 perfiles de preferencias y 20 nombres, y todos se repiten en un orden fijo. Es la huella de un generador que recorre listas cortas.

### 4.3 Las notas son uniformes (chi-cuadrado de bondad de ajuste)

**Idea:** en reseñas reales predominan las notas altas (4 y 5). Si cada nota de 1 a 5 aparece con la misma frecuencia, se parecen más a un dado que a opiniones.

- **H₀:** las cinco notas son igual de probables (20 % cada una).
- **H₁:** no lo son.

La prueba compara las frecuencias observadas con las esperadas (≈ 199,8 por nota): χ² = Σ (observado − esperado)² / esperado.

In [12]:
for nombre, notas in (("reseñas", resenas.Rating), ("historial", historial.ExperienceRating)):
    obs = notas.value_counts().sort_index()
    prueba = stats.chisquare(obs)        # por defecto compara contra frecuencias iguales
    print(f"{nombre:<10} frecuencias {obs.to_dict()}  χ² = {prueba.statistic:.2f}  p-valor = {prueba.pvalue:.2f}")

reseñas    frecuencias {1: 198, 2: 199, 3: 185, 4: 216, 5: 201}  χ² = 2.44  p-valor = 0.66
historial  frecuencias {1: 206, 2: 228, 3: 201, 4: 183, 5: 181}  χ² = 7.36  p-valor = 0.12


**Cómo leerlo:** los p-valores (≈ 0,66 y 0,12) son mayores que 0,05: **no se rechaza** que las notas sean uniformes. No prueba que lo sean, pero es lo contrario de lo que se ve en reseñas reales.

### 4.4 La nota no se relaciona con la popularidad (correlación)

**Idea:** en datos reales, los destinos mejor valorados tienden a tener mejores notas.

- **H₀:** no hay correlación entre `Popularity` y la nota (ρ = 0).
- **H₁:** sí la hay (ρ ≠ 0).

In [13]:
for nombre, df, col in (("reseñas", resenas, "Rating"), ("historial", historial, "ExperienceRating")):
    x = df.merge(destinos[["DestinationID", "Popularity"]], on="DestinationID")
    r, p = stats.pearsonr(x.Popularity, x[col])
    rho, p_s = stats.spearmanr(x.Popularity, x[col])
    print(f"{nombre:<10} Pearson r = {r:.3f} (p-valor = {p:.2f})   Spearman ρ = {rho:.3f} (p-valor = {p_s:.2f})")

reseñas    Pearson r = 0.044 (p-valor = 0.17)   Spearman ρ = 0.044 (p-valor = 0.17)
historial  Pearson r = 0.026 (p-valor = 0.41)   Spearman ρ = 0.028 (p-valor = 0.39)


**Cómo leerlo:** correlaciones de 0,03 a 0,04 con p-valores altos: la nota no tiene nada que ver con la popularidad del destino.

### 4.5 El texto de la reseña no se relaciona con el destino (chi-cuadrado de independencia)

**Idea:** «Loved the beaches!» debería aparecer en Goa, no en el Taj Mahal ni en las montañas de Leh Ladakh.

- **H₀:** el texto es independiente del tipo de destino.
- **H₁:** depende del tipo de destino.

In [14]:
x = resenas.merge(destinos[["DestinationID", "Type"]], on="DestinationID")
tabla = pd.crosstab(x.Type, x.ReviewText)
prueba = stats.chi2_contingency(tabla)
print(f"χ² = {prueba.statistic:.2f}, gl = {prueba.dof}, p-valor = {prueba.pvalue:.2f}")
tabla

χ² = 3.69, gl = 8, p-valor = 0.88


ReviewText,A historical wonder,Incredible monument!,Loved the beaches!
Type,,,
Adventure,68,70,80
Beach,65,60,65
City,62,72,57
Historical,70,66,63
Nature,68,65,68


**Cómo leerlo:** «Loved the beaches!» aparece 63 veces en destinos históricos y 80 en los de aventura, tanto o más que en las playas (65). El p-valor (≈ 0,88) no permite rechazar la independencia: el texto se asignó sin mirar el destino.

### 4.6 Las interacciones se reparten como el azar (Poisson)

**Idea:** en turismo real, unos pocos destinos famosos se llevan la mayoría de las visitas (cola larga). Si las 1998 interacciones se hubieran sorteado al azar entre 1000 destinos, el número de interacciones de cada destino seguiría una Poisson con media λ = 1998/1000 ≈ 2 (sección 3.1 del informe).

- **H₀:** las interacciones por destino siguen una Poisson(λ ≈ 2).
- **H₁:** no la siguen.

In [15]:
por_destino = inter.groupby("DestinationID").size().reindex(destinos.DestinationID, fill_value=0)
lam = por_destino.mean()
# Grupos 0, 1, …, 5 y «6 o más»: la prueba chi-cuadrado pide al menos 5 casos esperados por grupo,
# y «7 o más» solo espera ≈ 4,5, así que se junta con el 6.
k = np.arange(0, 6)
obs = np.append([(por_destino == v).sum() for v in k], (por_destino >= 6).sum())
esp = np.append(len(por_destino) * stats.poisson.pmf(k, lam), len(por_destino) * stats.poisson.sf(5, lam))
prueba = stats.chisquare(obs, esp, ddof=1)        # ddof=1 porque λ se estimó de los datos
print(f"λ = {lam:.3f}   χ² = {prueba.statistic:.2f}   gl = {len(obs) - 1 - 1}   p-valor = {prueba.pvalue:.3f}")
pd.DataFrame({"interacciones": [*map(str, k), "≥ 6"], "observados": obs, "esperados (Poisson)": esp.round(1)})

λ = 1.998   χ² = 0.27   gl = 5   p-valor = 0.998


,interacciones,observados,esperados (Poisson)
0,0,134,135.6
1,1,271,270.9
2,2,270,270.7
3,3,184,180.3
4,4,89,90.0
5,5,37,36.0
6,≥ 6,15,16.5


**Cómo leerlo:** los observados y los esperados casi coinciden, y el p-valor (≈ 0,998) es muy alto: no hay destinos «estrella». Los datos se ven exactamente como un sorteo.

### 4.7 Nadie reseñó un destino que visitó

**Idea:** en la vida real, una persona reseña un lugar porque lo visitó, así que muchos pares (usuario, destino) deberían estar en las dos tablas.

In [16]:
comunes = resenas.merge(historial, on=["UserID", "DestinationID"])
# Si los pares se hubieran sorteado al azar, ¿cuántos coincidirían? 999 reseñas × 999 visitas / (999 × 1000) pares posibles
print("Pares (usuario, destino) en ambas tablas:", len(comunes), "| esperados si todo fuera al azar: ≈", round(999 * 999 / (999 * 1000), 1))

Pares (usuario, destino) en ambas tablas: 0 | esperados si todo fuera al azar: ≈ 1.0


**Cómo leerlo:** no hay ningún par en común. Eso es incompatible con reseñas reales, que salen de visitas reales. (Ojo: si los pares se hubieran sorteado al azar, se esperaría ≈ 1 coincidencia, así que 0 tampoco es raro bajo el azar; la prueba muestra que las tablas se generaron **por separado**, no que haya un patrón oculto.)

### 4.8 La única relación débil: preferencias y tipo de destino (prueba binomial)

**Idea:** si las personas visitan lo que dicen preferir, la coincidencia debería ser alta. Cada perfil declara 2 de los 5 tipos, así que **por azar** coincidiría el 40 % de las interacciones.

- **H₀:** la coincidencia es la del azar (40 %).
- **H₁:** es mayor.

In [17]:
mapa = {"Beaches": "Beach", "Historical": "Historical", "City": "City", "Nature": "Nature", "Adventure": "Adventure"}
x = inter.merge(usuarios[["UserID", "Preferences"]], on="UserID")
coincide = [t in {mapa[p] for p in prefs.split(", ")} for t, prefs in zip(x.Type, x.Preferences)]
k_ok, n = int(np.sum(coincide)), len(coincide)
prueba = stats.binomtest(k_ok, n, 0.4, alternative="greater")
print(f"Coinciden {k_ok} de {n} = {100 * k_ok / n:.1f} %  (azar 40 %)  p-valor = {prueba.pvalue:.3f}")

Coinciden 846 de 1998 = 42.3 %  (azar 40 %)  p-valor = 0.017


**Cómo leerlo:** 42,3 % frente a 40 %. Sola, la prueba da p-valor ≈ 0,017; pero en el informe se probaron 6 hipótesis a la vez y, con la corrección de Bonferroni (6 × p-valor), deja de ser concluyente (sección 3.4). Es la única pista de una relación, y es muy débil.

## 5. Conclusión

| Prueba | Lo que esperaríamos en datos reales | Lo que hay |
|---|---|---|
| 4.1 Destinos | 1000 destinos distintos | 5 lugares que se repiten según (ID − 1) mod 5 |
| 4.2 Ciclos | Valores que no dependen del número de fila | Preferencias, textos y fechas en ciclos exactos de 3 |
| 4.3 Notas | Predominio de notas altas | Notas uniformes (no se rechaza H₀) |
| 4.4 Nota y popularidad | Alguna relación | Ninguna (r ≈ 0,03–0,04) |
| 4.5 Texto y destino | «Loved the beaches!» en playas | Texto independiente del destino |
| 4.6 Interacciones | Cola larga (destinos estrella) | Reparto de sorteo (Poisson) |
| 4.7 Reseñas e historial | Reseñas de lugares visitados | Tablas generadas por separado |
| 4.8 Preferencias | Coincidencia alta | Apenas por encima del azar, no concluyente |

Además, el autor escribe en Kaggle que el conjunto «was generated for educational and research purposes». Por eso el informe trata los datos como **sintéticos**: sirven para mostrar el método, no para sacar conclusiones sobre turistas reales.